# AKANE E9-1：辞書の確かめを、物1,681語ぜんぶの分類46個に広げる

このノートは **材料を取るだけ** です（比べは結果のファイルを読んで別に行います）。計画：sims/e9/PLAN_E9-1.md。

- E8と同じ辞書の鎖（Wiktionaryの説明文をたどる。途中の語の意味はGemmaが選ぶ）を、E4-2の物1,681語すべてに作ります。
- 鎖の点から46分類すべてへ「〜は〜の一種か」をNLIで判定します。正解はこのノートに入っていません。

## 手順
1. 「ランタイム」→「ランタイムのタイプを変更」→ **L4 GPU**。
2. 鍵マークの `HF_TOKEN` にアクセスを許可（E8と同じ）。
3. 「ランタイム」→「すべてのセルを実行」。ドライブの許可を聞かれたら許可。
4. 終わると、ドライブの `AKANE_アーキテクチャ/E9-1_results/E9-1_results.json` ができます。
5. 時間の見込み：Gemmaの意味選び 15〜25分、NLI 40〜70分。E8のダンプの控えがドライブにあれば、ダンプは取り直しません。
6. 途中で切れても、もう一度「すべてのセルを実行」すれば、済んだ所は飛ばします（途中の控えをドライブに保存しています）。

In [ ]:
import os, json, time, bz2, subprocess, sys
import numpy as np
from google.colab import drive, userdata
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/AKANE_アーキテクチャ"
OUT = f"{BASE}/E9-1_results"
os.makedirs(OUT, exist_ok=True)
NLI_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E6〜E8と同じ
GEMMA_ID = "google/gemma-2-2b-it"
DUMP_URL = "https://dumps.wikimedia.org/enwiktionary/latest/enwiktionary-latest-pages-articles.xml.bz2"
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=False)

In [ ]:
# 中身（sims/e8/e8_core.py と同じ）
"""E8の中身（Colabと手元の両方で使う）。
E7との違い：
- ダンプを読む時に、意味ごとの印（{{lb|en|slang}} など）を残す。途中の語の意味は8個まで持つ。
- 鎖の点を「語」ではなく「語＋意味の番号」にし、途中の語の意味を選ぶ方法を4通り：
  A 選ばない（E7と同じ、途中の語は最初の4つの意味を全部）
  B 印（古い・まれ・俗語など）のない最初の意味（Kovács 2022型）
  C 1つ前の説明文に一番近い意味（文の近さ、McIntosh 2009・OntoLearn型。分類Cは入れない）
  D Gemmaに「この説明文の中の語 g はどの意味か」を選ばせる（Meconi 2025・ConSeC型。分類Cは入れない）
- 各点から、46分類すべてへの「着き」をNLIで測る（打ち消し＝両立しない分類に着くか、に使う）。
"""
import re

MAX_DEFS_ENT, MAX_DEFS_MID, A_MID, DEPTH = 8, 8, 4, 3
BAD_LABELS = {"obsolete", "archaic", "dated", "rare", "slang", "vulgar", "offensive", "derogatory", "historical",
              "humorous", "euphemistic", "nonstandard", "pejorative", "ethnic slur", "slur", "obsolete outside dialects",
              "rare outside dialects", "nonce word", "jocular", "informal euphemism"}

# 頭の名詞（E7と同じ）
PERSON_HEADS = ("one who", "someone who", "somebody who", "a person who", "person who", "one that", "anyone who",
                "a person", "someone", "somebody")
OF_HEADS = {"kind", "type", "sort", "variety", "species", "form", "member", "one", "any", "piece", "item", "genus",
            "class", "breed", "style", "group", "family", "unit", "strain", "subspecies", "category", "instance",
            "example", "specimen", "individual", "portion", "quantity"}
META = {"synonym", "ellipsis", "term", "name", "word", "abbreviation", "plural", "spelling", "who", "something",
        "anything", "that", "which", "this", "one", "any", "form", "use", "sense", "variant", "misspelling", "acronym"}


def genus(defn, nlp):
    t = re.sub(r"\([^)]*\)", " ", defn).strip()
    low = t.lower().lstrip("“\"' ")
    for p in PERSON_HEADS:
        if low.startswith(p + " ") or low == p:
            return "person"
    doc = nlp(t[:200])
    chunks = list(doc.noun_chunks)
    if not chunks:
        return None
    root = chunks[0].root
    lem = root.lemma_.lower()
    if lem in OF_HEADS:
        for ch in root.children:
            if ch.dep_ == "prep" and ch.text.lower() == "of":
                objs = [c for c in ch.children if c.dep_ == "pobj"]
                if objs:
                    lem = objs[0].lemma_.lower()
                    break
    if lem == "being" and "human" in chunks[0].text.lower():
        lem = "human"
    if lem == "specie":
        lem = "species"
    if lem in META or not re.fullmatch(r"[a-z][a-z\-]*", lem):
        return None
    return lem


# --- ダンプの読み取り（E7と同じ書き換え。ただし lb の印は取っておく） ---
DROP_T = {"lb", "lbl", "label", "q", "qual", "qualifier", "i", "gloss", "gl", "sense", "s", "defdate", "rfdef", "rfclarify",
          "senseid", "tlb", "def-date", "c", "C", "topics", "cat", "rfex", "rfquote", "nb...", "ux", "uxi", "quote-book"}
ARG2_T = {"l", "m", "link", "mention", "ll", "l-self", "m-self", "cog"}
ARG1_T = {"w", "taxlink", "taxfmt", "vern", "ws", "pedia", "non-gloss", "n-g", "ngd", "non-gloss definition", "glossary",
          "smallcaps", "taxon", "spelink", "nowrap", "small", "sub", "sup"}
SKIP_LINE = ("plural of", "alternative form of", "alternative spelling of", "obsolete form of", "misspelling of",
             "abbreviation of", "initialism of", "acronym of", "obsolete spelling of", "archaic spelling of", "eye dialect of",
             "nonstandard spelling of", "synonym of", "clipping of", "short for", "ellipsis of", "contraction of")
SKIP_T = re.compile(r"^\{\{\s*(plural of|alternative form of|alt form|alt sp|alternative spelling of|obsolete form of|"
                    r"misspelling of|abbreviation of|abbr of|initialism of|acronym of|obsolete spelling of|archaic spelling of|"
                    r"eye dialect of|nonstandard spelling of|synonym of|syn of|clipping of|short for|ellipsis of|"
                    r"contraction of|form of|inflection of|en-past of)\s*\|", re.I)
LB = re.compile(r"\{\{\s*(?:lb|lbl|label|tlb)\s*\|\s*en\s*\|([^{}]*)\}\}")


def _tmpl(m):
    parts = [p.strip() for p in m.group(1).split("|")]
    name, args = parts[0].strip(), [p for p in parts[1:] if "=" not in p]
    if name in DROP_T:
        return ""
    if name in ARG2_T:
        return args[1] if len(args) > 1 else ""
    if name in ARG1_T:
        return args[0] if args else ""
    return ""


def strip_wikitext(t):
    for _ in range(6):
        t2 = re.sub(r"\{\{([^{}]*)\}\}", _tmpl, t)
        if t2 == t:
            break
        t = t2
    t = re.sub(r"\[\[(?:[^\]|]*\|)?([^\]]*)\]\]", r"\1", t)
    t = re.sub(r"'{2,}|<[^>]+>", "", t)
    return re.sub(r"\s+", " ", t).strip(" ;,.") + "."


def labels_of(raw):
    out = []
    for m in LB.finditer(raw):
        for p in m.group(1).split("|"):
            p = p.strip().lower()
            if p and "=" not in p and p not in ("_", "and", "or"):
                out.append(p)
    return out


def noun_senses_from_wikitext(wt, limit=MAX_DEFS_MID):
    """[[説明文, [印]], ...]（E7と同じ説明文の並び。印は {{lb|en|...}} から）"""
    m = re.search(r"(?ms)^==English==\s*$(.*?)(?=^==[^=]|\Z)", wt or "")
    if not m:
        return []
    out = []
    for sec in re.finditer(r"(?ms)^===+\s*Noun\s*===+\s*$(.*?)(?=^===|\Z)", m.group(1)):
        for line in sec.group(1).splitlines():
            if not line.startswith("# "):
                continue
            raw = line[2:].strip()
            if SKIP_T.match(raw):
                continue
            d = strip_wikitext(raw)
            if len(d) > 3 and not d.lower().startswith(SKIP_LINE):
                out.append([d[:220], labels_of(raw)])
    return out[:limit]


def parse_dump(fobj, log=print, every=500000):
    import xml.etree.ElementTree as ET
    out, n, title, ns = {}, 0, None, None
    for ev, el in ET.iterparse(fobj, events=("end",)):
        tag = el.tag.rsplit("}", 1)[-1]
        if tag == "title":
            title = el.text
        elif tag == "ns":
            ns = el.text
        elif tag == "text":
            if ns == "0" and title and re.fullmatch(r"[a-z][a-z\-]*", title) and el.text and "==English==" in el.text:
                ds = noun_senses_from_wikitext(el.text)
                if ds:
                    out[title] = ds
        elif tag == "page":
            n += 1
            el.clear()
            if n % every == 0:
                log(f"ページ {n:,}  名詞のある語 {len(out):,}")
    return out


def is_bad(labels):
    return any(l in BAD_LABELS for l in labels)


def first_good(senses):
    """B：印のない最初の意味の番号（全部印つきなら0）"""
    for k, (_, lb) in enumerate(senses):
        if not is_bad(lb):
            return k
    return 0


# --- 鎖をたどる ---
def build_lexicon(items, WKD, nlp, log=print):
    """L[w] = [{"def", "labels", "genus"}]。物の語は items の wk_defs（印なし）か WKD、途中の語は WKD から8個まで。
    A用の段（途中の語は最初の4つ全部）と、B/C/D用の段（途中の語は8個から1つ選ぶ）の両方で出てくる語をすべて集める。"""
    L = {}
    def senses(w, ent_defs=None):
        if w in L:
            return L[w]
        if ent_defs is not None:
            ss = [[d, []] for d in ent_defs][:MAX_DEFS_ENT]
        else:
            ss = WKD.get(w, [])[:MAX_DEFS_MID]
        L[w] = [{"def": d, "labels": lb, "genus": genus(d, nlp)} for d, lb in ss]
        return L[w]
    front = set()
    for it in items:
        ed = it.get("wk_defs")
        senses(it["ent"], ed if ed else None)
        front.add(it["ent"])
    for depth in range(DEPTH):
        nxt = set()
        for w in front:
            for x in L[w]:
                g = x["genus"]
                if g and g not in L:
                    senses(g)
                    nxt.add(g)
        log(f"段{depth + 1}: 新しい語 {len(nxt)}  語の合計 {len(L)}")
        front = nxt
    return L


def edges(L, ents):
    """(w,k) で genus g を持つ段のうち、たどり得るもの（物の語から3段まで、途中の語は全意味）"""
    seen, front, out = set(ents), set(ents), set()
    for _ in range(DEPTH):
        nxt = set()
        for w in front:
            for k, x in enumerate(L.get(w, [])):
                g = x["genus"]
                if g and g in L and g != w:
                    out.add((w, k))
                    if g not in seen:
                        seen.add(g); nxt.add(g)
        front = nxt
    return sorted(out)


def choice_prompt(w, d, g, cands):
    opts = "\n".join(f"{i + 1}. {c}" for i, c in enumerate(cands))
    return (f"Dictionary definition of \"{w}\": {d}\n\nIn this definition, which meaning of the word \"{g}\" is used?\n"
            f"{opts}\n\nAnswer with the number only.")


def premise(w, d):
    return f"\"{w}\" means: {d}"


def hyp(w, c):
    return f"{w[:1].upper() + w[1:]} is a type of {c}."


GROUP_WORDS = ["animal", "plant", "food", "man-made object", "material", "body part", "disease", "activity"]


def chain_nodes(L, x, method, choice=None):
    """物の語 x から3段までの点（語, 意味の番号）と段（(w,k)→(g,j)）。
    A：途中の語は最初の4つの意味を全部（E7と同じ）。B/C/D：途中の語は choice[(w,k)] で1つに決める。物の語は全部の意味。"""
    nodes, links = set(), set()
    front = [(x, k) for k in range(len(L.get(x, [])))]
    nodes.update(front)
    for _ in range(DEPTH):
        nxt = []
        for (w, k) in front:
            g = L[w][k]["genus"]
            if not g or g not in L or g == w or not L[g]:
                continue
            if method == "A":
                js = range(min(A_MID, len(L[g])))
            elif method == "B":
                js = [first_good([[s["def"], s["labels"]] for s in L[g]])]
            else:
                js = [choice.get((w, k), 0) if len(L[g]) > 1 else 0]
            for j in js:
                links.add(((w, k), (g, j)))
                if (g, j) not in nodes:
                    nodes.add((g, j)); nxt.append((g, j))
        front = nxt
    return nodes, links

In [ ]:
ENTS = json.loads(r'''["abalone", "abattoir", "abbess", "abbot", "abdominal", "aberrant", "abolitionist", "absentee", "abuser", "acacia", "accelerometer", "accomplice", "accordion", "accountant", "accused", "accuser", "achiever", "acrobat", "actress", "actuator", "adjuster", "adolescent", "adversary", "advertiser", "adviser", "agave", "agitator", "agua", "airbrush", "airbus", "aircraft", "airliner", "airplane", "airship", "alarmist", "alb", "albino", "alchemist", "alcoholic", "alderman", "ale", "aloe", "alternator", "althea", "alumnus", "ambulance", "amigo", "amplifier", "amputee", "anaconda", "analyzer", "anarchist", "ancestor", "android", "anesthesiologist", "ani", "anorexic", "ant", "antelope", "anthropologist", "anti", "apologist", "apparel", "appellant", "applicant", "applicator", "apprentice", "aquarium", "aquatic", "archbishop", "archdeacon", "archduke", "architect", "archivist", "argentine", "aristocrat", "armadillo", "armature", "armchair", "arsonist", "artist", "artiste", "ascot", "ashtray", "aspen", "aspirant", "assemblyman", "assistant", "asthmatic", "astrologer", "astronaut", "astronomer", "astrophysicist", "atheist", "athlete", "attacker", "attire", "auctioneer", "audiovisual", "aunt", "authoritarian", "autofocus", "avenger", "aviator", "ax", "ayatollah", "azalea", "baboon", "babu", "babysitter", "backpack", "backpacker", "badminton", "bailiff", "balaclava", "ballerina", "ballroom", "bandicoot", "bandit", "banjo", "bankrupt", "bantam", "barbell", "barge", "barometer", "baroness", "baronet", "barracuda", "barrister", "bartender", "basset", "bassist", "bassoon", "bathrobe", "bathtub", "batik", "batman", "battleship", "bayonet", "beagle", "beanie", "bedroom", "beekeeper", "beer", "beggar", "begum", "belle", "beloved", "belting", "benefactor", "bereaved", "beret", "berk", "betel", "bethel", "betrothed", "biceps", "bicycle", "bidet", "biennial", "bigot", "bilingual", "billionaire", "bimbo", "binoculars", "biochemist", "biographer", "biologist", "bisexual", "bison", "bistro", "biter", "blacksmith", "blaster", "blazer", "bleacher", "blogger", "bloodhound", "blouse", "boardroom", "boathouse", "boatman", "bobbin", "bobby", "bobcat", "bock", "bodice", "bodybuilder", "bonsai", "bookcase", "booker", "bookkeeper", "bookseller", "bookshelf", "borrower", "bot", "botanist", "boudoir", "bouillon", "bouncer", "bovine", "boyfriend", "brad", "bramble", "brandy", "brant", "brazier", "breadwinner", "breeder", "brew", "bricklayer", "bridesmaid", "briefcase", "brigadier", "brill", "broad", "brocade", "brogan", "bronco", "buddy", "buggy", "bulldog", "bulldozer", "bumblebee", "bungalow", "bureaucrat", "burglar", "burlap", "burqa", "burrito", "businessman", "businesswoman", "butch", "buttercup", "buttermilk", "buyer", "bystander", "cacao", "cactus", "caddie", "caddy", "cadet", "cafe", "cafeteria", "calender", "calico", "caliper", "caliph", "camcorder", "camel", "camellia", "cameraman", "campaigner", "candlestick", "cannibal", "cannonball", "canoe", "cantilever", "capacitor", "capo", "cappuccino", "captor", "carbine", "carburetor", "cardiologist", "caribou", "carpenter", "cartoonist", "cassette", "catamaran", "caterer", "cattle", "caveman", "celibate", "cellist", "cello", "centipede", "centrifuge", "centrist", "centurion", "cesspool", "chalet", "chalice", "chamomile", "chaplain", "chateau", "cheeseburger", "cheetah", "chef", "chiffon", "chime", "chimpanzee", "chipmunk", "chiropractor", "chisel", "choreographer", "chowder", "chronicler", "chronograph", "chrysalis", "chub", "chump", "churn", "cider", "circuitry", "citizen", "civilian", "claimant", "clairvoyant", "clamp", "clarence", "clarinet", "clarion", "clary", "classroom", "cleaver", "clergyman", "cleric", "clinician", "clitoris", "clock", "clockwork", "clover", "clubhouse", "coachman", "coauthor", "cobra", "cochin", "cockatoo", "cockroach", "coffin", "cofounder", "coleslaw", "collard", "colleen", "collider", "collie", "colonel", "colonial", "columbine", "columnist", "combatant", "commissar", "commodore", "commoner", "communicator", "compatriot", "composer", "compressor", "compulsive", "conch", "concierge", "concubine", "condom", "condor", "confessional", "confidant", "confidante", "conformist", "conger", "congressman", "conifer", "connoisseur", "conqueror", "conspirator", "consul", "consulate", "consumer", "contemplative", "contemporary", "contraceptive", "contrarian", "convalescent", "convener", "convert", "converter", "coordinator", "coot", "copycat", "copywriter", "corgi", "corkscrew", "cornerback", "cornet", "coroner", "corporal", "corrective", "corset", "corvette", "cosmopolitan", "cougar", "councilman", "countess", "coupe", "courtier", "cousin", "cowgirl", "crackpot", "crazy", "creditor", "cricketer", "criminal", "cripple", "critter", "crocodile", "croquet", "crossbow", "crucible", "crusher", "crustacean", "cubby", "cuckold", "cuddy", "cultivar", "cuppa", "curate", "curator", "curler", "curry", "cursor", "custard", "custodian", "customer", "cutler", "cutthroat", "cuttlefish", "cyborg", "cyclist", "cymbal", "dachshund", "dad", "daffodil", "dahlia", "daisy", "damsel", "dandelion", "daredevil", "darkroom", "daughter", "dauphin", "debtor", "debutante", "decadent", "decal", "deer", "defendant", "defibrillator", "deflector", "delegate", "delinquent", "dentist", "denture", "dependant", "depressive", "dermatologist", "descendant", "desk", "desperate", "detainee", "detonator", "diabetic", "dieter", "dildo", "dimmer", "dinghy", "dingo", "dinky", "dinosaur", "diocesan", "dirk", "disciple", "dispensary", "doge", "domestic", "dominatrix", "dominus", "dongle", "donna", "doorbell", "dowager", "dragonfly", "dragoon", "dramatist", "draper", "dreadnought", "dredge", "dropper", "drosophila", "drummer", "drunkard", "dry", "dubbing", "duce", "duchess", "dug", "dunce", "dyer", "dynamo", "earl", "earner", "easel", "eclectic", "ecologist", "economist", "educator", "eft", "egalitarian", "electric", "electrician", "electrode", "elitist", "emeritus", "emigrant", "emir", "emissary", "emitter", "employee", "employer", "empress", "empty", "entertainer", "entomologist", "entrepreneur", "environmentalist", "epileptic", "equatorial", "equine", "erica", "erotic", "escapist", "espresso", "essayist", "ethnic", "eunuch", "evergreen", "everlasting", "everyman", "exchanger", "executioner", "executor", "exhibitor", "expert", "exporter", "expressionist", "exterminator", "extremist", "extrovert", "eyewitness", "facilitator", "falafel", "falcon", "falconer", "fanatic", "fancier", "farmhouse", "fascist", "faucet", "fedora", "feline", "felt", "feminist", "fern", "fetus", "fiance", "fiancee", "fiduciary", "fife", "filly", "finalist", "financier", "finch", "fink", "firearm", "firstborn", "fisherman", "fitter", "fizz", "flamethrower", "flamingo", "flashlight", "flatmate", "flea", "flywheel", "foal", "footman", "forceps", "forklift", "freedman", "freelancer", "freeman", "friar", "fuchsia", "fulcrum", "fundamentalist", "galleon", "gallows", "gander", "gangsta", "gangster", "garment", "gasket", "gat", "gatehouse", "gavel", "gawker", "gazebo", "gazelle", "gearing", "gecko", "geezer", "geisha", "gelding", "geneticist", "genitalia", "geographer", "geologist", "georgette", "geranium", "giraffe", "girder", "gland", "glider", "gnu", "godmother", "godson", "goggles", "goldfish", "goldilocks", "golf", "golfer", "gorilla", "gosling", "governess", "grabber", "grader", "gramophone", "grandchild", "granddaughter", "grandfather", "grandma", "grandmaster", "grandparent", "grandson", "great", "greenhouse", "grenade", "greyhound", "gringo", "grizzly", "grocer", "grog", "groundhog", "groupie", "grubby", "guan", "guarantor", "guardsman", "guerrilla", "guesthouse", "guildhall", "guitar", "guitarist", "gymnast", "gynecologist", "gyroscope", "hacksaw", "haggis", "hairdresser", "hairpin", "hamster", "handcuff", "handkerchief", "handset", "handyman", "hangman", "harlequin", "harmonica", "harpsichord", "harrow", "hater", "hawthorn", "headdress", "headlight", "headliner", "headmaster", "headmistress", "headscarf", "headset", "hearse", "heathen", "heckler", "heifer", "heiress", "helicopter", "helmsman", "heterosexual", "hibiscus", "hiker", "hillbilly", "hippie", "hippopotamus", "historian", "hitchhiker", "hoarder", "hobbyist", "hod", "hoe", "hoist", "homeowner", "homophobe", "homosexual", "honeybee", "hooch", "hornet", "hosiery", "hostage", "hotel", "hourglass", "houseboat", "housekeeper", "housemate", "housewife", "hovercraft", "hoya", "huck", "hummingbird", "huntress", "hurling", "husband", "hyena", "hygienist", "hypnotist", "hypocrite", "ibis", "idealist", "idiot", "illiterate", "illustrator", "imam", "immigrant", "immune", "impala", "imperialist", "impersonator", "implant", "importer", "imposter", "impressionist", "incompetent", "incubator", "incumbent", "incurable", "individualist", "industrialist", "infernal", "infielder", "informer", "inga", "inhabitant", "inhaler", "injector", "innocent", "inpatient", "inquirer", "insider", "insomniac", "insured", "integrator", "intellectual", "interceptor", "interferometer", "intern", "interviewee", "interviewer", "intestine", "introvert", "intruder", "invader", "invalid", "inventor", "invertebrate", "inverter", "investor", "ironclad", "islander", "isolationist", "itinerant", "ivy", "jackal", "jaeger", "jaguar", "jammer", "janitor", "jasmine", "jeep", "jester", "jib", "jodhpur", "jogger", "juggler", "jukebox", "juror", "juvenile", "kangaroo", "kayak", "kea", "kennel", "keyboardist", "khaki", "kicker", "kidnapper", "kidney", "kilt", "kimono", "kingfisher", "kinsman", "kirk", "kitchen", "kitten", "knitwear", "koala", "kook", "krill", "kris", "kylie", "laborer", "lacrosse", "ladle", "ladybug", "laird", "lally", "lancer", "landlady", "landlord", "landowner", "lantern", "laptop", "larva", "larynx", "laser", "lass", "lathe", "latrine", "laureate", "lawyer", "layman", "leaker", "legate", "legislator", "lemonade", "lemur", "lender", "leotard", "letterman", "liana", "liar", "liberator", "libertine", "librarian", "licensee", "lidar", "lifeboat", "lifeguard", "lilac", "lily", "limiter", "limousine", "lingerie", "lioness", "liqueur", "literate", "livestock", "liza", "llama", "lobbyist", "locator", "locket", "locksmith", "locomotive", "lodger", "loner", "longer", "longhorn", "loom", "loudspeaker", "loyalist", "luge", "lung", "lyre", "lyricist", "macadamia", "machete", "machinist", "macho", "madame", "madder", "maestro", "magdalen", "maggot", "magistrate", "magneto", "magnum", "maharaja", "mahatma", "mailman", "mako", "malik", "mallard", "mamba", "mammal", "mammoth", "manatee", "mandolin", "manger", "mangrove", "mantel", "mantis", "mantua", "marauder", "marc", "margarita", "margate", "marigold", "mariner", "mariposa", "marksman", "marlin", "married", "marten", "martini", "masjid", "masseuse", "mastiff", "mastodon", "matador", "matchbox", "matchmaker", "mater", "materiel", "mathematician", "maul", "maxi", "mayor", "meatball", "mediator", "megaphone", "melancholic", "menial", "mensch", "mentor", "mercenary", "merchant", "merino", "mesquite", "messenger", "meteorologist", "metronome", "microfilm", "microphone", "microprocessor", "microscope", "midge", "midshipman", "midwife", "migrant", "militant", "milkman", "millionaire", "milo", "mimic", "miner", "minion", "minivan", "minnow", "minster", "miser", "misfit", "misogynist", "missus", "moa", "mockingbird", "mod", "modem", "modernist", "molester", "moll", "mollie", "monastery", "mongoose", "moped", "moray", "morgue", "mosque", "mosquito", "motel", "moth", "motorboat", "motorcycle", "motorcyclist", "mountaineer", "mulatto", "murderer", "murine", "musket", "muslin", "mustang", "mystic", "nacho", "namesake", "narc", "narcissist", "narrator", "national", "nawab", "necktie", "negotiator", "nematode", "neoliberal", "nephew", "netball", "nettle", "neurologist", "neuroscientist", "neurosurgeon", "neurotic", "neutral", "newlywed", "newscaster", "newsreader", "newt", "niece", "nigger", "nightgown", "nightshade", "niqab", "nomad", "nondescript", "nonpartisan", "notary", "novelist", "nudist", "nunnery", "nutter", "oboe", "obsessive", "obstetrician", "oligarch", "ombudsman", "omelet", "oncologist", "ophthalmologist", "opportunist", "oppressor", "optimist", "optometrist", "orangutan", "orator", "orchid", "organist", "originator", "ornamental", "orthodontist", "oryx", "oscillator", "osprey", "outcast", "outerwear", "outhouse", "outpatient", "overlord", "overseer", "owl", "pacifist", "padlock", "paella", "pagoda", "paintbrush", "paisley", "paleontologist", "palmetto", "palmyra", "pancreas", "panelist", "pantry", "pantyhose", "panzer", "papillon", "parachute", "paralegal", "paramedic", "paranoid", "paraplegic", "paratrooper", "parka", "parsonage", "pasha", "passenger", "passerby", "pater", "patriot", "pauper", "payer", "pedestrian", "pedophile", "pelican", "pendulum", "penguin", "penis", "penitent", "pensioner", "peony", "peregrine", "perennial", "perfectionist", "performer", "peripheral", "periscope", "perpetrator", "pervert", "peshmerga", "pessimist", "petticoat", "pew", "pharmacist", "philanthropist", "photographer", "photojournalist", "physicist", "physiologist", "phytoplankton", "pia", "pianist", "piccolo", "pickpocket", "pigeon", "piglet", "pika", "pimp", "pinto", "pistol", "pitchfork", "pituitary", "pizza", "placental", "plaintiff", "platypus", "playboy", "playhouse", "playmaker", "playmate", "pleaser", "plenipotentiary", "plover", "plow", "plumber", "plush", "poet", "poi", "policeman", "polygraph", "poncho", "ponderosa", "pontifical", "pooch", "poodle", "poppy", "porcupine", "porpoise", "porridge", "portable", "postmaster", "potion", "potter", "practitioner", "prankster", "preacher", "prefect", "presbytery", "preteen", "priestess", "primrose", "prince", "princess", "prior", "priory", "prisoner", "private", "probable", "proctor", "prodigal", "professor", "progenitor", "programmer", "proletarian", "propagandist", "propeller", "prosecutor", "prospector", "prosthesis", "prostitute", "protectionist", "protege", "provost", "prowler", "prude", "psychiatrist", "psychic", "psychologist", "psychotherapist", "psychotic", "publican", "publicist", "puffin", "pug", "pulley", "pullover", "pupa", "puppeteer", "purifier", "purist", "pursued", "purveyor", "quartermaster", "quitter", "racehorse", "racist", "radar", "radiologist", "raglan", "raincoat", "rancher", "rani", "rapier", "rapporteur", "ratatouille", "ratchet", "rattlesnake", "raven", "rayon", "razor", "reactionary", "receptionist", "recliner", "redneck", "reeve", "refugee", "registrant", "religious", "repairman", "reporter", "reptile", "reset", "resistor", "restaurant", "restaurateur", "retailer", "retiree", "retriever", "reversible", "revisionist", "rhesus", "rheumatic", "rhinoceros", "rhododendron", "rickey", "ridley", "rifle", "ringleader", "rink", "ripper", "risotto", "rival", "roadhouse", "robber", "rodent", "roebuck", "rogue", "roommate", "rosita", "rowan", "rudd", "rugby", "rustic", "sabra", "sadist", "sagebrush", "sahib", "sailboat", "salad", "salesman", "salesperson", "sandal", "sandwich", "sarcophagus", "sari", "sarin", "sashimi", "satchel", "satin", "satirist", "sauerkraut", "sauna", "saxophonist", "scalpel", "scapegoat", "scarab", "schizophrenic", "schmuck", "schoolboy", "schoolgirl", "schoolteacher", "scientist", "scion", "scold", "scraper", "screenwriter", "scrim", "scythe", "seaplane", "searchlight", "secessionist", "sectarian", "sectional", "seedling", "segregate", "seller", "semaphore", "senator", "senna", "separatist", "serge", "serviceman", "sesame", "settle", "shaheed", "shaman", "shawl", "shed", "sheeting", "sheriff", "sherry", "shill", "shim", "ship", "shirt", "shiv", "shogun", "shook", "shopkeeper", "shotgun", "shredder", "shrine", "shrub", "sibling", "sickle", "sieve", "signor", "silly", "simian", "simpleton", "simulator", "singlet", "sinner", "sissy", "skateboard", "skater", "skeptic", "skewer", "ski", "skier", "skiff", "skinhead", "skyscraper", "slacker", "sled", "sleuth", "slingshot", "slipknot", "slob", "sloop", "smuggler", "snapdragon", "sniffer", "sniper", "snob", "snoop", "snowboarder", "snowmobile", "soccer", "socialist", "socialite", "sociologist", "sociopath", "sofa", "solenoid", "soloist", "sonar", "songbird", "songwriter", "sophomore", "sorcerer", "sorceress", "soundtrack", "sovereign", "sow", "spacecraft", "spandex", "spaniel", "spastic", "speakeasy", "speedboat", "speedometer", "spelt", "sphincter", "spitfire", "spitz", "spokesman", "spokesperson", "spokeswoman", "sportswear", "spouse", "springbok", "sprinkler", "sprinter", "sputnik", "squaw", "stable", "stagecoach", "stakeholder", "stallion", "stanhope", "stapler", "starfish", "starling", "starship", "statesman", "stator", "steakhouse", "steamboat", "steed", "steeper", "stepdaughter", "stepfather", "stepmother", "stepson", "stethoscope", "stewardess", "stickler", "stiletto", "stingray", "stockbroker", "stockholder", "stockman", "stole", "stoner", "stopwatch", "storeroom", "stork", "stowaway", "strainer", "strapless", "strategist", "stray", "streetcar", "strider", "stupid", "sturgeon", "subcontractor", "succulent", "suiting", "suitor", "sultan", "sunflower", "supercharger", "supercomputer", "supermodel", "supplier", "supremacist", "surfer", "surgeon", "surrealist", "survivalist", "sushi", "swami", "swan", "swatch", "sweatshirt", "swimsuit", "swine", "switcher", "swivel", "sword", "swot", "synagogue", "synchrotron", "syringe", "tabor", "tactician", "tadpole", "tailor", "takin", "tam", "tambourine", "tamer", "tapeworm", "tartan", "taster", "tavern", "taxpayer", "teammate", "teapot", "techie", "telegraph", "telescope", "temp", "tenement", "tennis", "tequila", "teriyaki", "termite", "tern", "terrier", "terrorist", "tetra", "theologian", "theorist", "therapist", "thermometer", "thermos", "thermostat", "thief", "thistle", "thwart", "tiara", "tidy", "tights", "timepiece", "toby", "toddler", "toga", "tomahawk", "tomboy", "tongs", "toolbox", "toothpick", "tortoise", "totalitarian", "tourist", "towel", "tracker", "trader", "traditionalist", "trainee", "trampoline", "transducer", "transformer", "transistor", "transponder", "transvestite", "trapeze", "trapper", "traveler", "tray", "treasurer", "triceps", "tricycle", "trident", "tripod", "trombone", "troublemaker", "trusty", "tugboat", "tulip", "tulle", "tun", "turbine", "turtleneck", "tweeter", "typewriter", "typist", "uke", "undercarriage", "underdog", "undergraduate", "underpants", "undersecretary", "understudy", "underwear", "undesirable", "undies", "unfortunate", "uniform", "untouchable", "urchin", "urologist", "usurper", "uterus", "utilitarian", "vagrant", "valedictorian", "valerian", "valet", "vaporizer", "vase", "vassal", "vegan", "vegetarian", "velvet", "vermouth", "veronica", "vertebrate", "veterinarian", "vigilante", "villager", "vine", "violin", "violinist", "viper", "vise", "visitor", "vizier", "vodka", "vole", "voter", "voyager", "voyeur", "vulva", "waitress", "wallaby", "wallet", "wally", "walrus", "wanton", "warden", "warlord", "warrior", "warship", "washroom", "watchmaker", "watchman", "waterfowl", "waveguide", "wearer", "weatherman", "webcam", "weevil", "welder", "westerner", "wether", "wheelchair", "whiskey", "whisperer", "whitey", "widow", "widower", "wife", "wildflower", "winch", "windowsill", "winger", "wingman", "wisteria", "womanizer", "wombat", "woodcock", "woodpecker", "wop", "workaholic", "workbench", "workman", "workstation", "wormwood", "worthy", "wraparound", "wrestler", "wristwatch", "yacht", "yarrow", "yucca", "zebra", "zoologist"]''')
CATS = json.loads(r'''["living thing", "animal", "mammal", "bird", "fish", "insect", "reptile", "dog", "plant", "tree", "flower", "fruit", "vegetable", "food", "dish", "drink", "man-made object", "tool", "device", "machine", "equipment", "vehicle", "car", "boat", "aircraft", "weapon", "container", "furniture", "clothing", "footwear", "toy", "musical instrument", "building", "room", "person", "worker", "artist", "athlete", "material", "fabric", "metal", "body part", "organ", "disease", "game", "sport"]''')
ITEMS = [{'ent': w, 'wk_defs': None} for w in ENTS]   # 物の語の説明文もダンプから
print(len(ENTS), '語', len(CATS), '分類')

## 1. Wiktionaryの意味（E8の控えを使う。なければダンプを取る：25〜45分）

In [ ]:
cands = [f"{OUT}/E9-1_wiktionary_dump_senses.json", f"{BASE}/E8_results/E8_wiktionary_dump_senses.json"]
got = [p for p in cands if os.path.exists(p)]
if got:
    WKD = json.load(open(got[0])); print("ドライブから読み込み:", got[0], len(WKD))
else:
    dump = "/content/enwiktionary.xml.bz2"
    subprocess.run(["wget", "-c", "-q", "--show-progress", "-O", dump, DUMP_URL], check=True)
    t = time.time()
    with bz2.open(dump, "rb") as f:
        WKD = parse_dump(f)
    print("読み終わり:", len(WKD), "語", f"{time.time() - t:.0f}秒")
    json.dump(WKD, open(cands[0], "w"), ensure_ascii=False)
print("説明文のある物の語の割合:", np.mean([w in WKD for w in ENTS]).round(3))

## 2. 語の意味の一覧と、意味を選ぶ必要のある段

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner"])
L = build_lexicon(ITEMS, WKD, nlp)
EDGES = edges(L, ENTS)
multi = [(w, k) for w, k in EDGES if len(L[L[w][k]["genus"]]) > 1]
print("語:", len(L), " 段:", len(EDGES), " 意味を選ぶ必要のある段:", len(multi))

## 3. 途中の語の意味をGemmaに選ばせる（GPU、15〜25分、途中の控えあり）

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
cpath = f"{OUT}/E9-1_choice_D.json"
CHOICE_D = {tuple(k[:2]): k[2] for k in json.load(open(cpath))} if os.path.exists(cpath) else {}
todo = [x for x in multi if x not in CHOICE_D]
print("済み:", len(CHOICE_D), " 残り:", len(todo))
if todo:
    tok = AutoTokenizer.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), padding_side="left")
    gm = AutoModelForCausalLM.from_pretrained(GEMMA_ID, token=userdata.get("HF_TOKEN"), torch_dtype=torch.float16, device_map="cuda").eval()
    NUM = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1, MAX_DEFS_MID + 1)]
    t = time.time(); B = 16
    for i in range(0, len(todo), B):
        batch = todo[i:i + B]
        qs = [choice_prompt(w, L[w][k]["def"], L[w][k]["genus"], [x["def"] for x in L[L[w][k]["genus"]]]) for w, k in batch]
        ps = [tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True) for q in qs]
        enc = tok(ps, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            last = gm(**enc, logits_to_keep=1).logits[:, -1, :].float()
        for (w, k), row in zip(batch, last):
            n = len(L[L[w][k]["genus"]])
            CHOICE_D[(w, k)] = int(row[NUM[:n]].argmax())
        if (i // B) % 100 == 0:
            print(f"{i}/{len(todo)}  {time.time() - t:.0f}秒")
            json.dump([[w, k, v] for (w, k), v in CHOICE_D.items()], open(cpath, "w"))
    json.dump([[w, k, v] for (w, k), v in CHOICE_D.items()], open(cpath, "w"))
    del gm; torch.cuda.empty_cache()
print("最初の意味以外を選んだ割合:", np.mean([v != 0 for v in CHOICE_D.values()]).round(3))

## 4. 鎖を作り、NLIで判定する（GPU、40〜70分、途中の控えあり）

In [ ]:
CHAINS, nodes_all = {}, set()
for w in ENTS:
    nodes, links = chain_nodes(L, w, "D", CHOICE_D)
    CHAINS[w] = (sorted(nodes), sorted(links)); nodes_all |= set(nodes)
NODES = sorted(nodes_all)
STEPS = sorted({(w, k) for w, k in NODES if L[w][k]["genus"]})
print("点:", len(NODES), " 段:", len(STEPS), " 着きの組:", len(NODES) * len(CATS))
from transformers import AutoModelForSequenceClassification
ntok = AutoTokenizer.from_pretrained(NLI_ID)
nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID, torch_dtype=torch.float16).cuda().eval()
lab = {v.lower(): int(k) for k, v in nm.config.id2label.items()}
ORDER = [lab["entailment"], lab["contradiction"]]
def nli_probs(P, H, B=128):
    out = np.zeros((len(P), 2), np.float32)
    for i in range(0, len(P), B):
        enc = ntok(P[i:i + B], H[i:i + B], return_tensors="pt", padding=True, truncation=True, max_length=192).to("cuda")
        with torch.no_grad():
            pr = torch.softmax(nm(**enc).logits.float(), -1)[:, ORDER]
        out[i:i + len(pr)] = pr.cpu().numpy()
    return out
S = nli_probs([premise(w, L[w][k]["def"]) for w, k in STEPS], [hyp(w, L[w][k]["genus"]) for w, k in STEPS])
print("段 済み")
# 着き：点ごとに46分類。2,000点ずつ控えを保存
CH_N = 2000
A = np.zeros((len(NODES), len(CATS), 2), np.float32)
t = time.time()
for s in range(0, len(NODES), CH_N):
    p = f"{OUT}/E9-1_arrive_{s}.npy"
    part = NODES[s:s + CH_N]
    if os.path.exists(p) and np.load(p).shape[0] == len(part):
        A[s:s + len(part)] = np.load(p); continue
    P = [premise(w, L[w][k]["def"]) for w, k in part for C in CATS]
    H = [hyp(w, C) for w, k in part for C in CATS]
    r = nli_probs(P, H).reshape(len(part), len(CATS), 2)
    np.save(p, r); A[s:s + len(part)] = r
    print(f"着き {s + len(part)}/{len(NODES)}  {time.time() - t:.0f}秒")
res = {"meta": {"nli": NLI_ID, "gemma": GEMMA_ID, "dump": DUMP_URL, "date": time.strftime("%Y-%m-%d"),
                "n_ents": len(ENTS), "cats": CATS},
       "L": L,
       "choice_D": [[w, k, v] for (w, k), v in CHOICE_D.items()],
       "chains": {w: [[list(x) for x in nd], [[list(a), list(b)] for a, b in lk]] for w, (nd, lk) in CHAINS.items()},
       "steps": [[w, k, round(float(p[0]), 4), round(float(p[1]), 4)] for (w, k), p in zip(STEPS, S)],
       "nodes": [list(x) for x in NODES],
       "arrive_e": np.round(A[:, :, 0], 4).tolist(),
       "arrive_c": np.round(A[:, :, 1], 4).tolist()}
path = f"{OUT}/E9-1_results.json"
json.dump(res, open(path, "w"), ensure_ascii=False)
print("保存:", path, round(os.path.getsize(path) / 1e6, 1), "MB")